# 5 · Ejercicio 4: abstraer un patrón (20 min)

El enunciado te da 2-3 funciones "basadas en un mismo patrón de diseño" y pide:
- **a)** una función de orden superior que encapsule el patrón,
- **b)** reescribir las funciones usándola (una línea cada una).

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen.

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() => empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

## La receta

1. Pon las funciones una al lado de otra. Marca lo **igual** (la estructura) y lo que **cambia**.
2. Lo que cambia → **parámetro función** (`f: A => B`, `f: (A, B) => C`…).
3. Los tipos concretos (`Int`, `String`…) → genéricos (`A`, `B`, `C`).
4. Copia la estructura llamando a `f` donde había código concreto. En la llamada recursiva pasa también `f`.
5. Cada función original = la HOF + una lambda.

Pon la estructura en una lista de parámetros y las funciones en otra: `def hof[A, B](datos: ...)(f: ...)`.

## Ejemplo (examen 25-26) · `mapLeft`

In [ ]:
def add1(t: Either[Int, String]): Either[Int, String] =
    t match
        case Left(i)  => Left(i + 1)
        case Right(s) => Right(s)

def not(t: Either[Boolean, Char]): Either[Boolean, Char] =
    t match
        case Left(b)  => Left(!b)
        case Right(c) => Right(c)

Igual: el `match`; el `Right` se devuelve tal cual. Cambia: lo que se le hace al valor del `Left` (`i + 1`, `!b`) → `f: A => C`.

In [ ]:
def mapLeft[A, B, C](t: Either[A, B])(f: A => C): Either[C, B] =
    t match
        case Left(a)  => Left(f(a))
        case Right(b) => Right(b)

def add1(t: Either[Int, String]): Either[Int, String] = mapLeft(t)(_ + 1)
def not(t: Either[Boolean, Char]): Either[Boolean, Char] = mapLeft(t)(!_)

(add1(Left(1)), not(Left(true)), add1(Right("x")))

## Las cuatro familias que salen

| Familia | Idea | Signatura típica |
|---|---|---|
| **map** | misma forma, se transforma el contenido | `def map[A, B](t: F[A])(f: A => B): F[B]` |
| **fold** | se recorre y se calcula un resultado | `def fold[A, B](t: F[A])(base: B)(combinar: ...): B` |
| **unfold** | se **genera** desde una semilla | `def unfold[A, B](seed: B)(next: B => Option[(A, B)]): List[A]` |
| **traverse** | como map, pero `f` puede fallar (`Option`); si falla una, falla todo | `def traverse[A, B](l: List[A])(f: A => Option[B]): Option[List[B]]` |

### Ejemplo de familia fold: árboles

In [ ]:
// Patrón común de size, sum, depth... sobre árboles
def foldT[A, B](t: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    t match
        case Empty() => empty
        case Node(l, x, r) => node(foldT(l)(empty)(node), x, foldT(r)(empty)(node))

def size[A](t: Tree[A]): Int = foldT(t)(0)((l, _, r) => l + 1 + r)
def sumT(t: Tree[Int]): Int  = foldT(t)(0)((l, x, r) => l + x + r)

val t = node(leaf(1), 2, right(3, leaf(4)))
(size(t), sumT(t))

### Ejemplo de familia traverse

In [ ]:
def traverse[A, B](l: List[A])(f: A => Option[B]): Option[List[B]] =
    l match
        case Nil => Some(Nil)
        case head :: tail =>
            (f(head), traverse(tail)(f)) match
                case (Some(b), Some(bs)) => Some(b :: bs)
                case _ => None

def parse(l: List[String]): Option[List[Int]] = traverse(l)(_.toIntOption)

(parse(List("1", "2")), parse(List("1", "x")))

## Mini-ejercicio (5 min)

Abstrae el patrón en `mapOption` y reimplementa ambas funciones.

In [ ]:
def inc(o: Option[Int]): Option[Int] =
    o match
        case None => None
        case Some(n) => Some(n + 1)

def len(o: Option[String]): Option[Int] =
    o match
        case None => None
        case Some(s) => Some(s.length)

In [ ]:
def mapOption[A, B](o: Option[A])(f: A => B): Option[B] =
    ???

<details>
<summary><b>Ver solución</b></summary>

```scala
def mapOption[A, B](o: Option[A])(f: A => B): Option[B] =
    o match
        case None => None
        case Some(a) => Some(f(a))

def inc(o: Option[Int]): Option[Int] = mapOption(o)(_ + 1)
def len(o: Option[String]): Option[Int] = mapOption(o)(_.length)
```

</details>

## ¡Listo!

Ahora haz un simulacro con reloj: `examenes-prueba/Prueba1-Enunciado.ipynb` (1h30m) y corrige con `Prueba1-Solucion.ipynb`. Si te atascas en algo, el detalle está en `claudetemario`.